# 04 - LLM con Few-Shot Prompting

Probamos el enfoque LLM (Claude / OpenAI) con few-shot prompting para ABSA.

**Requisitos:**
- Variable de entorno `ANTHROPIC_API_KEY` o `OPENAI_API_KEY`.
- Paquetes `anthropic` u `openai` instalados.

In [ ]:
import os
import sys
from pathlib import Path


def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("No se encontró la raíz del repositorio")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.aspects.extractor import extract_aspects, load_spacy_model
from src.data.loader import load_reviews
from src.llm.prompting import analyze_with_llm, build_default_client


## 1. Construir cliente

In [ ]:
RUN_LLM = os.getenv("RUN_LLM") == "1"
MAX_LLM_REVIEWS = int(os.getenv("MAX_LLM_REVIEWS", "5"))

if RUN_LLM:
    client = build_default_client()
    print(type(client).__name__)
else:
    client = None
    print("LLM remoto desactivado. Define RUN_LLM=1 y una API key para ejecutar esta celda con costo.")


## 2. Extraer aspectos y analizar

In [ ]:
df = load_reviews(REPO_ROOT / "data/sample/reviews_sample.csv")
nlp = load_spacy_model()

results = []
if client is None:
    print("Sin llamadas remotas; results queda vacío.")
else:
    for _, row in df.head(MAX_LLM_REVIEWS).iterrows():
        text = row["text"]
        aspects = extract_aspects(text, nlp)
        sentiments = analyze_with_llm(text, aspects, client)
        results.append({"text": text, "aspects": sentiments})
results


## 3. Comparar con etiquetas del lexicón

Útil para detectar dónde el LLM y el línea base discrepan.

In [ ]:
from src.classical.lexicon import load_lexicon, score_aspect_lexicon

lex = load_lexicon()
to_lbl = lambda s: "pos" if s > 0.1 else "neg" if s < -0.1 else "neu"

if not results:
    print("No hay resultados LLM para comparar.")
else:
    for r in results:
        print(r["text"])
        for asp, lbl in r["aspects"].items():
            lex_lbl = to_lbl(score_aspect_lexicon(r["text"], asp, lex))
            marker = " " if lex_lbl == lbl else "*"
            print(f"  {marker} {asp:>15} | LLM={lbl} | Lex={lex_lbl}")
        print()


## 4. Coste y latencia

Documentar tokens consumidos y tiempo por reseña antes de escalar a miles.